In [8]:
import optuna
import numpy as np

In [9]:
# ============================
# B-сплайновый базис (Кокс-де Бор)
# ============================

def make_knot_vector(a, b, n_internal, degree=3):
    """Расширенный вектор узлов (clamped)."""
    internal = np.linspace(a, b, n_internal + 2)
    return np.concatenate([
        np.repeat(a, degree), internal, np.repeat(b, degree)
    ])


def bspline_basis(x, knots, degree=3):
    """Матрица B-сплайнового базиса (N x K)."""
    x = np.asarray(x, dtype=float)
    knots = np.asarray(knots, dtype=float)
    N = len(x)

    B = np.zeros((N, len(knots) - 1))
    for i in range(len(knots) - 1):
        if knots[i] < knots[i + 1]:
            B[(x >= knots[i]) & (x < knots[i + 1]), i] = 1.0
    for i in range(len(knots) - 2, -1, -1):
        if knots[i] < knots[i + 1]:
            B[x == knots[-1], i] = 1.0
            break

    for d in range(1, degree + 1):
        B_new = np.zeros((N, len(knots) - 1 - d))
        for i in range(len(knots) - 1 - d):
            d1 = knots[i + d] - knots[i]
            d2 = knots[i + d + 1] - knots[i + 1]
            if d1 > 0:
                B_new[:, i] += (x - knots[i]) / d1 * B[:, i]
            if d2 > 0:
                B_new[:, i] += (knots[i + d + 1] - x) / d2 * B[:, i + 1]
        B = B_new
    return B


def diff_matrix(K, order=2):
    """Матрица конечных разностей порядка order."""
    D = np.eye(K)
    for _ in range(order):
        D = D[1:, :] - D[:-1, :]
    return D


# ============================
# 2D тензорный P-сплайн
# ============================

def pspline_2d_fit(x, y, Z, n_basis=30, degree=3, penalty_order=2, lam=1.0, x_range=None, y_range=None):
    """
    Подгонка 2D P-сплайна (тензорное произведение).
    Возвращает коэффициенты A и узловые вектора kx, ky.
    """
    if x_range is None:
        x_range = (x[0], x[-1])
    if y_range is None:
        y_range = (y[0], y[-1])

    kx = make_knot_vector(x_range[0], x_range[1], n_basis, degree)
    ky = make_knot_vector(y_range[0], y_range[1], n_basis, degree)
    Bx = bspline_basis(x, kx, degree)
    By = bspline_basis(y, ky, degree)

    Kx, Ky = Bx.shape[1], By.shape[1]

    Dx = diff_matrix(Kx, penalty_order)
    Dy = diff_matrix(Ky, penalty_order)

    BxTBx = Bx.T @ Bx
    ByTBy = By.T @ By
    DxTDx = Dx.T @ Dx
    DyTDy = Dy.T @ Dy

    rhs = (Bx.T @ Z @ By).ravel()

    LHS = (np.kron(ByTBy, BxTBx)
           + lam * np.kron(np.eye(Ky), DxTDx)
           + lam * np.kron(DyTDy, np.eye(Kx)))

    alpha = np.linalg.solve(LHS, rhs)
    A = alpha.reshape(Kx, Ky)

    return A, kx, ky


def pspline_2d_eval(A, kx, ky, x_eval, y_eval, degree=3):
    """Вычисление P-сплайна на сетке."""
    Bx = bspline_basis(x_eval, kx, degree)
    By = bspline_basis(y_eval, ky, degree)
    return Bx @ A @ By.T

In [10]:
Z_raw = np.load("./data/fuji_elevation.npy").astype(np.float64)
lat = np.load("./data/fuji_lat.npy")
lon = np.load("./data/fuji_lon.npy")

Z_raw = Z_raw[::-1, :]
lat_sn = lat[::-1]

X_coords = lon
Y_coords = lat_sn

step_points = 8

X_pts = X_coords[::step_points]
Y_pts = Y_coords[::step_points]
Z_pts = Z_raw[::step_points, ::step_points]

n_basis = 40

In [18]:
def objective(trial):
    lam = trial.suggest_float("lam", 0.01, 1)

    A, kx, ky = pspline_2d_fit(
        X_pts, Y_pts, Z_pts,
        n_basis=n_basis, degree=3, penalty_order=2, lam=lam,
        x_range=(X_coords[0], X_coords[-1]),
        y_range=(Y_coords[0], Y_coords[-1])
    )
    Z_interp_on_raw = pspline_2d_eval(A, kx, ky, X_coords, Y_coords, degree=3)

    error = Z_interp_on_raw - Z_raw
    mae = np.nanmean(np.abs(error))

    return mae

In [19]:
study = optuna.create_study(direction="minimize")  # решение на минимум
study.optimize(objective, n_trials=100)

study.best_params

[I 2026-05-03 21:07:49,652] A new study created in memory with name: no-name-9172b22d-3448-40e0-a4bb-cc3f86f3cc33
[I 2026-05-03 21:07:49,870] Trial 0 finished with value: 10.88201109523189 and parameters: {'lam': 0.05856950467228022}. Best is trial 0 with value: 10.88201109523189.
[I 2026-05-03 21:07:50,185] Trial 1 finished with value: 11.02774706122283 and parameters: {'lam': 0.06869202290611921}. Best is trial 0 with value: 10.88201109523189.
[I 2026-05-03 21:07:50,714] Trial 2 finished with value: 12.973247328028043 and parameters: {'lam': 0.3167190847320575}. Best is trial 0 with value: 10.88201109523189.
[I 2026-05-03 21:07:51,061] Trial 3 finished with value: 13.555886583011695 and parameters: {'lam': 0.45083123968225813}. Best is trial 0 with value: 10.88201109523189.
[I 2026-05-03 21:07:51,299] Trial 4 finished with value: 13.898014475387434 and parameters: {'lam': 0.5479653518764674}. Best is trial 0 with value: 10.88201109523189.
[I 2026-05-03 21:07:51,524] Trial 5 finished 

{'lam': 0.010119678286647735}